# 3교시 · LLM Agent로 수집 코드 만들고 정제하기

**이 시간에 할 일**

- 실습 A — 같은 API에 대해 프롬프트를 세 단계로 바꿔가며 생성된 코드의 차이를 봅니다
- 실습 B — 응답을 표준 스키마로 옮기고, 계약 위반을 잡습니다
- 실습 C — 오류와 예외를 자동 처리합니다

**이 노트북은 네트워크 없이 전부 돌아갑니다.** 장애 상황을 일부러 만들어 넣어야 하기 때문에, 실제 API 대신 가짜 응답기를 씁니다.

## 3-0. 오늘의 제1원칙

> **언어모델에게 숫자를 읽게 하지 말고, 숫자를 읽는 코드를 짜게 하십시오.**

같은 데이터를 두 방식으로 다루면 이렇게 갈립니다.

| | 값을 읽히기 | 코드를 짜게 하기 |
|---|---|---|
| 재현성 | 매번 다를 수 있음 | 같은 입력에 같은 출력 |
| 검증 | 원문 대조 외에 방법 없음 | 단위 테스트 가능 |
| 비용 | 행 수에 비례 | 한 번 |
| 틀렸을 때 | 조용히 틀림 | 예외로 드러남 |

**언어모델이 개입할 자리는 코드를 만드는 단계이지, 값을 다루는 단계가 아닙니다.**

---

### 실패 모드 네 가지

오후 내내 이걸 방어합니다.

1. **조용한 실패** — 0건을 받았는데 정상 종료
2. **부분 수집** — 첫 쪽만 받고 끝
3. **스키마 드리프트** — 응답 키 이름이 바뀌어 값이 통째로 사라짐
4. **무한 재시도** — 실패한 요청을 끝없이 반복

## 3-1. 가짜 응답기

장애를 일부러 만들어야 코드의 차이가 드러납니다. 실제 API 대신 이걸 씁니다.

In [ ]:
import json, random, time
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone, timedelta
import pandas as pd

KST = timezone(timedelta(hours=9))


class FakeResponse:
    def __init__(self, status, payload=None):
        self.status_code = status
        self._payload = payload
    def json(self):
        if self._payload is None:
            raise ValueError("본문이 JSON이 아닙니다")
        return self._payload
    def raise_for_status(self):
        if self.status_code >= 400:
            raise RuntimeError(f"HTTP {self.status_code}")


def make_rows(country, years, per_page, page, drift=False):
    key = "iso3" if drift else "countryiso3code"     # 드리프트 시 키 이름이 바뀜
    all_rows = [{
        "indicator": {"id": "SP.DYN.TFRT.IN"},
        key: country,
        "date": str(y),
        "value": None if y == 2019 else round(1.3 - 0.05 * (y - 2015), 3),
    } for y in years]
    start = (page - 1) * per_page
    return all_rows, all_rows[start:start + per_page]


class FakeAPI:
    """시나리오에 따라 장애를 주입하는 가짜 API."""
    def __init__(self, scenario="normal"):
        self.scenario = scenario
        self.calls = 0

    def get(self, url, params=None, timeout=None):
        self.calls += 1
        params = params or {}
        page = int(params.get("page", 1))
        per_page = int(params.get("per_page", 50))
        years = list(range(2015, 2026))

        if self.scenario == "rate_limit" and self.calls <= 2:
            return FakeResponse(429)
        if self.scenario == "flaky" and self.calls == 1:
            raise TimeoutError("연결 시간 초과")
        if self.scenario == "empty":
            return FakeResponse(200, [{"page": 1, "pages": 1, "per_page": per_page, "total": 0}, []])
        if self.scenario == "html":
            return FakeResponse(200, None)

        drift = (self.scenario == "drift")
        paged = (self.scenario == "paged")
        pp = 4 if paged else per_page
        all_rows, chunk = make_rows("KOR", years, pp, page, drift)
        meta = {"page": page, "pages": (len(all_rows) + pp - 1) // pp,
                "per_page": pp, "total": len(all_rows)}
        return FakeResponse(200, [meta, chunk])


SCENARIOS = ["normal", "flaky", "rate_limit", "empty", "paged", "drift", "html"]
print("시나리오:", ", ".join(SCENARIOS))

## 실습 A · 프롬프트 3단 진화

같은 API, 같은 목표. 프롬프트만 바꿉니다.

### 1단계 — "이 API 호출하는 코드 짜줘"

가장 흔한 요청입니다. 나오는 코드도 대체로 이렇습니다.

In [ ]:
def collect_v1(session):
    """1단계 산출물. 성공 경로만 있습니다."""
    r = session.get("https://api.example/v2/indicator", params={"format": "json"})
    payload = r.json()
    rows = payload[1]
    return pd.DataFrame([{
        "region_code": x.get("countryiso3code"),   # 키가 없어도 조용히 None
        "period": int(x["date"]),
        "value": x["value"],
    } for x in rows])


# 정상 상황에서는 잘 돕니다
collect_v1(FakeAPI("normal")).head()

### 2단계 — 명세서와 출력 스키마를 함께 줍니다

> "아래 명세서를 참고해서 수집 코드를 짜줘. 출력은 이 컬럼과 타입으로 고정해줘."

재사용 가능한 모양이 나옵니다. 하지만 아직 장애는 못 견딥니다.

In [ ]:
COLUMNS = ["source", "indicator_code", "region_code", "period", "value", "unit"]

def collect_v2(session):
    """2단계 산출물. 출력 스키마가 고정됩니다."""
    r = session.get("https://api.example/v2/indicator",
                    params={"format": "json", "per_page": 500})
    r.raise_for_status()
    payload = r.json()
    rows = payload[1]
    out = pd.DataFrame([{
        "source": "WORLDBANK",
        "indicator_code": x["indicator"]["id"],
        "region_code": x["countryiso3code"],
        "period": int(x["date"]),
        "value": x["value"],
        "unit": "명",
    } for x in rows])
    return out[COLUMNS]


collect_v2(FakeAPI("normal")).head(3)

### 3단계 — 실패 케이스를 먼저 나열시킵니다

> **"이 API를 호출할 때 일어날 수 있는 실패를 다섯 가지 나열하고, 그 다섯 가지를 각각 처리하는 코드를 짜줘."**

프롬프트에 이 한 줄을 넣는 것만으로 재시도, 백오프, 페이지 처리, 스키마 확인이 알아서 들어옵니다.

**요구사항을 나열하는 것보다 실패를 나열하게 하는 쪽이 훨씬 견고한 코드를 만듭니다.**

In [ ]:
def collect_v3(session, max_retries=3, base_delay=0.01, verbose=False):
    """3단계 산출물. 다섯 가지 실패를 각각 처리합니다."""
    collected, page = [], 1
    meta_total = None

    while True:
        # 실패 1: 일시적 오류와 호출 제한 → 재시도와 백오프. 횟수를 제한해 무한 반복을 막음
        for attempt in range(max_retries):
            try:
                r = session.get("https://api.example/v2/indicator",
                                params={"format": "json", "per_page": 500, "page": page},
                                timeout=20)
                if r.status_code == 429:
                    raise RuntimeError("호출 제한")
                r.raise_for_status()
                break
            except Exception as e:
                if attempt == max_retries - 1:
                    raise RuntimeError(f"{max_retries}회 재시도 후 실패: {e}")
                time.sleep(base_delay * (2 ** attempt))
                if verbose: print(f"  재시도 {attempt+1}회 ({e})")

        # 실패 2: 본문이 JSON이 아님 (점검 페이지, 로그인 리다이렉트 등)
        try:
            payload = r.json()
        except Exception:
            raise RuntimeError("응답 본문이 JSON이 아닙니다. 원문을 확인하세요.")

        # 실패 3: 최상위 구조가 예상과 다름
        if not (isinstance(payload, list) and len(payload) == 2):
            raise RuntimeError(f"응답 구조가 명세와 다릅니다: {type(payload).__name__}")

        meta, rows = payload[0], payload[1]
        meta_total = int(meta.get("total", 0))

        # 실패 4: 스키마 드리프트. 필요한 키가 없으면 조용히 비우지 말고 멈춤
        if rows:
            need = {"date", "value", "indicator"}
            region_key = next((k for k in ("countryiso3code", "iso3") if k in rows[0]), None)
            missing = need - rows[0].keys()
            if missing or region_key is None:
                raise RuntimeError(f"응답 키가 바뀌었습니다. 누락 {missing or ''} 지역키 {region_key}")
            collected += [{
                "source": "WORLDBANK",
                "indicator_code": x["indicator"]["id"],
                "region_code": x[region_key],
                "period": int(x["date"]),
                "value": x["value"],
                "unit": "명",
            } for x in rows]

        if page >= int(meta.get("pages", 1)):
            break
        page += 1

    # 실패 5: 조용한 실패. 0건인데 정상 종료하는 것을 막음
    if meta_total == 0 or not collected:
        raise RuntimeError("수집 결과가 0건입니다. 조건을 확인하세요.")
    if len(collected) != meta_total:
        raise RuntimeError(f"부분 수집: {len(collected)}건 / 전체 {meta_total}건")

    return pd.DataFrame(collected)[COLUMNS]


collect_v3(FakeAPI("normal")).head(3)

### 세 버전을 같은 장애에 넣어봅니다

여기가 이 시간의 핵심 셀입니다.

In [ ]:
def try_run(fn, scenario, full=11):
    """행 수만 보지 않습니다. 조용히 망가진 흔적까지 같이 봅니다."""
    try:
        df = fn(FakeAPI(scenario))
    except Exception as e:
        return f"중단: {str(e)[:40]}"

    flags = []
    if len(df) < full:
        flags.append(f"{full - len(df)}행 유실")
    if "region_code" in df.columns and df["region_code"].isna().all():
        flags.append("지역 전부 소실")
    return f"{len(df)}행" + (f" ⚠ {', '.join(flags)}" if flags else " 정상")


table = []
for s in SCENARIOS:
    table.append({
        "시나리오": s,
        "1단계": try_run(collect_v1, s),
        "2단계": try_run(collect_v2, s),
        "3단계": try_run(collect_v3, s),
    })

pd.set_option("display.max_colwidth", 46)
pd.DataFrame(table)

### 결과를 어떻게 읽어야 하는가

**"중단"이 좋은 결과입니다.**

- `empty` — 1단계는 0행을 돌려주고 끝납니다. 아무도 모릅니다. 3단계는 멈춥니다
- `paged` — 1·2단계는 첫 쪽만 받고 정상 종료합니다. **분석은 반쪽 데이터로 진행됩니다**
- `drift` — 응답 키가 바뀌면 1·2단계는 예외로 죽거나 값을 잃습니다. 3단계는 원인을 말하고 멈춥니다
- `flaky`, `rate_limit` — 일시적 오류인데 1·2단계는 그냥 죽습니다

가장 위험한 건 예외가 나는 경우가 아니라 **숫자가 나오는데 틀린 경우**입니다. `empty`와 `paged`가 그렇습니다.

> 자동화의 목표는 실패하지 않는 것이 아니라, **실패했을 때 조용하지 않은 것**입니다.

## 실습 B · 표준 스키마와 계약

수집 코드가 여러 개로 늘어나면 출력 모양이 제각각이 됩니다. 그래서 **계약을 코드로 박아둡니다.**

In [ ]:
MISSING_CODES = {
    "NA_NOTSURVEYED":   "미조사",
    "NA_NOTAPPLICABLE": "해당없음",
    "NA_CONFIDENTIAL":  "비공개",
}

@dataclass
class Observation:
    source: str
    indicator_code: str
    region_code: str
    period: int
    value: float | None
    unit: str
    vintage: str
    retrieved_at: str
    source_url: str
    missing_reason: str | None = None

    def __post_init__(self):
        if not (1900 <= self.period <= 2100):
            raise ValueError(f"period 범위 밖: {self.period}")
        if self.value is None and self.missing_reason not in MISSING_CODES:
            raise ValueError(f"결측인데 사유가 없습니다: region={self.region_code} period={self.period}")
        if self.value is not None and not isinstance(self.value, (int, float)):
            raise ValueError(f"value 타입 오류: {type(self.value).__name__}")
        if self.vintage not in ("확정", "잠정", "추계"):
            raise ValueError(f"vintage 값 오류: {self.vintage}")


ok = Observation("WORLDBANK", "SP.DYN.TFRT.IN", "KOR", 2024, 0.75, "명",
                 "확정", datetime.now(KST).isoformat(), "https://api.worldbank.org/...")
print("정상 생성:", ok.region_code, ok.period, ok.value)

try:
    Observation("WORLDBANK", "SP.DYN.TFRT.IN", "KOR", 2019, None, "명",
                "확정", datetime.now(KST).isoformat(), "https://...")
except ValueError as e:
    print("차단됨 →", e)

### 결측에 사유를 강제하는 이유

`value`가 비어 있으면 **왜 비었는지를 반드시 적게** 만들었습니다.

이걸 강제하지 않으면 나중에 셋이 구분되지 않습니다.

- 미조사는 보간을 검토할 수 있습니다
- 해당없음은 보간하면 안 됩니다
- 비공개는 값이 존재하므로 다른 경로를 찾아볼 수 있습니다

빈칸 하나로 뭉갠 뒤에는 어느 쪽이었는지 복원할 방법이 없습니다. **수집 시점에만 알 수 있는 정보라서 그때 잡아야 합니다.**

In [ ]:
def to_observations(raw_rows, source, source_url, vintage="확정"):
    now = datetime.now(KST).isoformat()
    out, rejected = [], []
    for x in raw_rows:
        try:
            v = x["value"]
            out.append(Observation(
                source=source,
                indicator_code=x["indicator"]["id"],
                region_code=x.get("countryiso3code") or x.get("iso3"),
                period=int(x["date"]),
                value=v,
                unit="명",
                vintage=vintage,
                retrieved_at=now,
                source_url=source_url,
                missing_reason=None if v is not None else "NA_NOTSURVEYED",
            ))
        except Exception as e:
            rejected.append({"원본": x, "사유": str(e)})
    return out, rejected


payload = FakeAPI("normal").get("https://api.example/v2/indicator", params={"per_page": 500}).json()
obs, rejected = to_observations(payload[1], "WORLDBANK", "https://api.worldbank.org/...")

print(f"통과 {len(obs)}건 / 거부 {len(rejected)}건")
df = pd.DataFrame([asdict(o) for o in obs])
df[["region_code", "period", "value", "missing_reason", "vintage"]].head(6)

## 실습 C · 검증 세트

계약은 한 행씩 봅니다. 검증은 **데이터셋 전체**를 봅니다.

핵심은 각 검사에 **실패했을 때 어떻게 할지**를 같이 정하는 것입니다.

In [ ]:
def validate(df, expected_regions=None, expected_years=None):
    results = []

    def rule(name, ok, gate, detail=""):
        results.append({"검사": name, "결과": "통과" if ok else ("확인" if gate == "기록" else "실패"),
                        "실패 시": gate, "비고": detail})

    rule("행 수 0 아님", len(df) > 0, "중단", f"{len(df)}행")

    need = {"source", "indicator_code", "region_code", "period", "value", "unit", "vintage"}
    miss = need - set(df.columns)
    rule("필수 컬럼 존재", not miss, "중단", f"누락 {miss}" if miss else "")

    dup = df.duplicated(subset=["indicator_code", "region_code", "period"]).sum()
    rule("중복 키 없음", dup == 0, "중단", f"중복 {dup}건")

    vals = df["value"].dropna()
    out_of_range = ((vals < 0) | (vals > 3)).sum()
    rule("값 범위 0~3", out_of_range == 0, "중단", f"이탈 {out_of_range}건")

    if expected_regions:
        got = set(df["region_code"])
        missing_r = set(expected_regions) - got
        rule("지역 전수 존재", not missing_r, "중단", f"누락 {missing_r}" if missing_r else "")

    if expected_years:
        missing_y = set(expected_years) - set(df["period"])
        rule("기간 연속성", not missing_y, "중단", f"누락 연도 {sorted(missing_y)}" if missing_y else "")

    s = df.dropna(subset=["value"]).sort_values("period")
    chg = s["value"].pct_change().abs()
    spikes = int((chg > 0.30).sum())
    rule("전년 대비 변화율 30% 이내", spikes == 0, "기록",
         f"초과 {spikes}건 — 원문 대조 필요" if spikes else "")

    return pd.DataFrame(results)


report = validate(df, expected_regions=["KOR"], expected_years=range(2015, 2026))
report

### "실패 시" 열이 실제로 동작해야 합니다

검사 결과를 사람이 눈으로 보고 넘어가면 아무 의미가 없습니다. **중단 조건은 코드가 강제**해야 합니다.

In [ ]:
def gate(report_df):
    blocking = report_df[(report_df["결과"] == "실패") & (report_df["실패 시"] == "중단")]
    if len(blocking):
        raise RuntimeError(
            "검증 실패로 파이프라인을 중단합니다:\n"
            + "\n".join(f"  - {r['검사']}: {r['비고']}" for _, r in blocking.iterrows())
        )
    warn = report_df[report_df["결과"] == "확인"]
    return {"통과": int((report_df["결과"] == "통과").sum()),
            "확인": len(warn), "실패": 0}


print("정상 데이터:", gate(report))

# 일부러 망가뜨려 봅니다
broken = df.copy()
broken.loc[broken.index[:3], "value"] = 9.9
try:
    gate(validate(broken, expected_regions=["KOR"], expected_years=range(2015, 2026)))
except RuntimeError as e:
    print("\n" + str(e))

---

## 정리

**만든 것**

- 실패를 나열시켜 얻은 수집 코드 — 재시도, 페이지 처리, 스키마 확인, 조용한 실패 탐지
- 행 단위 계약 — 결측 사유를 강제하는 데이터 구조
- 데이터셋 단위 검증 — 중단 조건이 코드로 강제됨

**가져가실 것 세 가지**

1. **"실패 다섯 가지를 먼저 나열하고 그걸 처리하는 코드를 짜줘"** — 프롬프트 한 줄의 차이가 큽니다
2. **결측 사유는 수집 시점에만 알 수 있습니다** — 그때 안 적으면 영원히 모릅니다
3. **중단 조건은 사람이 아니라 코드가 지킵니다**

지금까지는 노트북 안에서 조각으로 만들었습니다. 4교시에는 이걸 **다시 실행할 수 있는 하나의 파이프라인**으로 묶고, 웹 수집 데이터와 내부 자료를 붙입니다.